In [ ]:
## This Notebook is meant to be a sandbox for testing out the nfl_data_py library and exploring the data it provides. 
## It is not meant to be a polished analysis, but rather a place to experiment and learn how to use the library effectively.

import nfl_data_py as nfl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)

# Check for a local copy of the schedule data. If does not exist, create one
# This is a large file, so we want to avoid downloading it multiple times, and this should improve performance

if os.path.exists('.\data\schedules.csv'):
    print("Local copy of schedule data found. Loading from file...")
    schedules = pd.read_csv('.\data\schedules.csv')
else:
    schedules = nfl.import_schedules(list(range(1999, 2025)))
    os.makedirs('.\data', exist_ok=True)
    schedules.to_csv('.\data\schedules.csv', index=False)
    print("Schedules downloaded and saved locally")

In [ ]:
# Starting small with just the 2025 season to explore the data and get a feel for it
# Will do basic analysis to confirm with seperate source like PFR that outputs are correct

schedule = nfl.import_schedules([2025])

print(f"Rows: {schedule.shape[0]}, Columns: {schedule.shape[1]}")
schedule.head()

In [ ]:
print(schedule.columns.tolist())

In [ ]:

neededCols = [
    'game_id', 'season', 'game_type', 'week',
    'home_team', 'away_team',
    'home_score', 'away_score',
    'result', 'home_rest', 'away_rest',
    'div_game', 'location'
]

filteredSchedules = schedules[neededCols]

vikingsSchedule = filteredSchedules[(filteredSchedules['home_team'] == 'MIN') | (filteredSchedules['away_team'] == 'MIN')].copy()
print(f"Total Vikings Games: {len(vikingsSchedule)}")
print(vikingsSchedule.head())

In [ ]:
# Tag when the Vikings have a bye week
# If the had_bye column is TRUE, then the Vikings had a bye week in the previous week
vikingsSchedule['had_bye'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['home_rest'] >= 14) |
 (vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['away_rest'] >= 14))

#Check to see how many bye games we found
# For 2014 - 2025, we should expect to see 1 bye game per season, so 12 total bye games
print(f"Total Vikings Games with Bye Week: {vikingsSchedule['had_bye'].sum()}")
print(f"Total games without a bye week: {(~vikingsSchedule['had_bye']).sum()}")

In [ ]:
vikingsSchedule[vikingsSchedule['had_bye'] == True].groupby('season')['had_bye'].count()

In [ ]:
# There are some incorrect values in the bye week count. Adjusting rest day filter to 13, and also filtering for just the regular season
vikingsSchedule = filteredSchedules[((filteredSchedules['home_team'] == 'MIN') | (filteredSchedules['away_team'] == 'MIN')) & (filteredSchedules['game_type'] == 'REG')].copy()
vikingsSchedule['had_bye'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['home_rest'] >= 13) |
 (vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['away_rest'] >= 13))

print(f"Total Vikings Games with Bye Week: {vikingsSchedule['had_bye'].sum()}")
print(f"Total games without a bye week: {(~vikingsSchedule['had_bye']).sum()}")
print("Bye Week count by season:")
print(vikingsSchedule[vikingsSchedule['had_bye'] == True].groupby('season')['had_bye'].count())

In [ ]:
# Add a column for whether the Vikings won the game or not
vikingsSchedule['vikings_win'] = ((vikingsSchedule['home_team'] == 'MIN') & (vikingsSchedule['result'] > 0)) | ((vikingsSchedule['away_team'] == 'MIN') & (vikingsSchedule['result'] < 0))
print(f"# of Vikings Wins: {vikingsSchedule['vikings_win'].sum()}")

In [ ]:
# First Question: How often do the Vikings win their first game back from a bye week?

# Find Total Games played, and total games won
totalGames = len(vikingsSchedule)
totalWins = vikingsSchedule['vikings_win'].sum()
overallWinPercentage = totalWins / totalGames if totalGames > 0 else 0
print(f"Total Vikings Games: {totalGames}")
print(f"Total Vikings Wins: {totalWins}")
print(f"Overall Vikings Win Percentage: {overallWinPercentage:.2%}")
print("-----------------------------------")

postByeGames = vikingsSchedule[vikingsSchedule['had_bye'] == True]
totalByeGames = len(postByeGames)
gamesWon = postByeGames['vikings_win'].sum()
winPercentage = postByeGames['vikings_win'].mean() if totalByeGames > 0 else 0
print(f"Total Games After Bye Week: {totalByeGames}")
print(f"Total Games Won After Bye Week: {gamesWon}")
print(f"Vikings Win Percentage After Bye Week: {winPercentage:.2%}")
print("-----------------------------------")

noneByeGames = vikingsSchedule[vikingsSchedule['had_bye'] == False]
totalNonByeGames = len(noneByeGames)
nonByeWins = noneByeGames['vikings_win'].sum()
nonByeWinPercentage = noneByeGames['vikings_win'].mean() if totalNonByeGames > 0 else 0
print(f"Total Games Without Bye Week: {totalNonByeGames}")
print(f"Total Games Won Without Bye Week: {nonByeWins}")
print(f"Vikings Win Percentage Without Bye Week: {nonByeWinPercentage:.2%}")

In [ ]:
winRateData = pd.DataFrame({
    'Category': ["Overall Win Percentage", "Win Percentage After Bye Week", "Win Percentage Without Bye Week"],
    'Win Percentage': [overallWinPercentage, winPercentage, nonByeWinPercentage]
})

print(winRateData)

In [ ]:
# Make a bar chart to compare the win percentages

# Initialize the plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Category', y='Win Percentage', data=winRateData, color='purple')

plt.title('Vikings Win Percentage Comparison')
plt.ylabel('Win Percentage')
plt.xlabel('')

# Format the Y-axis to show percentages
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

# Properly scale the Y-axis to show from 0% to 80%
plt.ylim(0, 0.80)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Let's enhance the bar chart by adding the exact win percentage values on top of each bar for better clarity

# Initialize the plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Category', y='Win Percentage', data=winRateData, color='purple')

plt.title('Vikings Win Percentage Bye Week vs Non-Bye Week')
plt.ylabel('Win Percentage')
plt.xlabel('Game Type')

# Add the percentage values on top of each bar
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{bar.get_height():.2%}',
        ha='center',
        va='bottom'
    )

# Format the Y-axis to show percentages
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

# Properly scale the Y-axis to show from 0% to 80%
plt.ylim(0, 0.80)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Begin point differential analysis

# Add a point differential column to the df
vikingsSchedule['differential'] = np.where(vikingsSchedule['home_team'] == 'MIN', vikingsSchedule['home_score'] - vikingsSchedule['away_score'],
                                                                                  vikingsSchedule['away_score'] - vikingsSchedule['home_score'])

vikingsSchedule.head()

In [ ]:
overallDifferential = vikingsSchedule['differential'].mean()
avgByeWeekDifferential = vikingsSchedule[vikingsSchedule['had_bye'] == True]['differential'].mean()
avgNonByeWeekDifferential = vikingsSchedule[vikingsSchedule['had_bye'] == False]['differential'].mean()

print(f"Overall Average Point Differential: {overallDifferential:.2f}")
print(f"Average Point Differential After Bye Week: {avgByeWeekDifferential:.2f}")
print(f"Average Point Differential Without Bye Week: {avgNonByeWeekDifferential:.2f}")

In [ ]:
# Plot the differnetial on a bar chart
differentialData = pd.DataFrame({
    'Category': ["Overall", "After Bye Week", "Without Bye Week"],
    'Point Differential': [overallDifferential, avgByeWeekDifferential, avgNonByeWeekDifferential]
})

plt.figure(figsize=(10, 6))
colors = ["green" if x > 0 else "red" for x in differentialData['Point Differential']]
ax = sns.barplot(x='Category', y='Point Differential', data=differentialData, hue='Category', palette=colors, legend=False)

# Set the labels
plt.title("Vikings Point Differential Comparison, 2014-2024")
plt.ylabel("Average Point Differential")
plt.xlabel("")

# If the differential is > 0, the bar is green, if < 0, the bar should be red
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{bar.get_height():.2f}",
        ha='center',
        va='bottom'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Plot the HOme and away wins after the bye week
gameData = pd.DataFrame({
    'Category' : ["Total", "Total", "Home", "Home", "Away", "Away"],
    'Type' : ["Games", "Wins", "Games", "Wins", "Games", "Wins"],
    'Value': [len(postByeHomeGames) + len(postByeAwayGames),  # Total games played
              postByeWins + postByeAwayWins,                  # Total Games won
              len(postByeHomeGames),                          # Home Games played
              postByeWins,                                    # Home Games won
              len(postByeAwayGames),                          # Away Games played
              postByeAwayWins]                                # Away Games won
})
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Category', y='Value', data=gameData, hue='Type', palette=['purple', 'gold'], legend=True)

# Set the labels
plt.title("Vikings Home vs Away Bye Games")
plt.ylabel("# Games / Wins")
plt.xlabel("")

# If the differential is > 0, the bar is green, if < 0, the bar should be red
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{bar.get_height()}",
        ha='center',
        va='bottom'
    )

plt.tight_layout()
plt.show()

In [ ]:
# Plot the HOme and away wins after the bye week, as winning percentages
gameData = pd.DataFrame({
    'Category' : ["All Bye Games", "Home Games", "Away Games"],
    'Value' : [winPercentage, postByeHomeWinPerecentage, postByeAwayWinPercentage]                          # Away Games won
})
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Category', y='Value', hue='Category', data=gameData, palette=['purple', 'gold'], legend=False)

# Set the labels
plt.title("Vikings Home vs Away Bye Games")
plt.ylabel("Winning Percentage")
plt.xlabel("")

# If the differential is > 0, the bar is green, if < 0, the bar should be red
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{bar.get_height():.2%}',
        ha='center',
        va='bottom'
    )

# Format the Y-axis to show percentages
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
allPostByeGames = vikingsSchedule[vikingsSchedule['had_bye'] == True].copy()
allPostByeGames = allPostByeGames.sort_values(by='season')
allPostByeGames['cumulative_wins'] = allPostByeGames['vikings_win'].cumsum()

print(allPostByeGames[['season', 'week', 'vikings_win', 'cumulative_wins']])

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(allPostByeGames['season'], allPostByeGames['cumulative_wins'], marker="o")
# hint: x axis = season, y axis = cumulative_wins
# use plt.plot() instead of sns.barplot() for a line chart
# plt.plot(x, y, marker='o') draws the line with dots at each data point

plt.title("Vikings Cumulative Post-Bye Wins 1999-2024")
plt.xlabel("Season")
plt.ylabel("Cumulative Wins")

plt.show()